링크: https://www.kaggle.com/competitions/santander-customer-satisfaction/overview$0

# 1. 주제: Santander Customer Satisfaction

# 2. 데이터
## **(1) train.csv**
- Santander 고객의 금융 관련 정보를 담고 있는 학습 데이터
- ID : 각 고객을 구분하기 위한 고유 식별자
- 다양한 ind_var, num_var, saldo_var 등 370개의 익명화된 고객 관련 feature
- TARGET : 고객 만족 여부를 나타내는 타깃 변수. 0 → 만족 고객 / 1 → 불만족 고객

## **(2) test.csv**
- 모델을 이용하여 고객 만족 여부를 예측하기 위한 테스트 데이터
- ID : 고객 식별자
- train.csv와 동일한 형태의 370개 feature

## **(3) sample_submission.csv**
- Kaggle에 결과를 제출할 때 사용하는 샘플 제출 파일

# 3. 코드 흐름
## **(1) 데이터 확인 및 탐색**
- 먼저 train.csv와 test.csv를 pandas를 이용하여 불러온 후 데이터의 크기와 feature 구성을 확인
- 학습 데이터 35,000개, 테스트 데이터 15,000개, feature는 각각 370개로 구성되어 있음을 확인

## **(2) Constant Feature(상수형 변수) 제거**
- 첫 번째 Feature Selection 방법으로 VarianceThreshold를 사용. 상수형 변수(Constant Feature)는 모든 관측치에서 동일한 값을 가지기 때문에 고객의 만족 여부를 구분하는 데 어떠한 정보도 제공하지 않음
- VarianceThreshold(threshold=0)을 사용하여 분산이 0인 변수를 찾음. 그 결과 전체 370개 feature 중 51개가 상수형 변수인 것을 확인
- 따라서 51개의 불필요한 변수를 제거하여 370개 → 319개 feature로 차원을 축소

## **(3) Quasi-constant Feature(준상수형 변수) 제거**
- 준상수형 변수는 모든 값이 완전히 동일하지는 않지만, 데이터의 대부분이 하나의 값으로 구성되어 있어 모델 학습에 거의 정보를 제공하지 않는 변수
- VarianceThreshold(threshold=0.01)을 적용하여 분산이 매우 낮은 변수를 제거
- 그 결과 370개의 feature 중 107개의 변수가 quasi-constant feature로 확인되었으며, 이를 제거하여 최종적으로 263개 feature를 유지

## **(4) Univariate Feature Selection**
- 각 feature와 target 사이의 통계적인 관계를 이용하여 중요한 feature를 선택하는 방법
- 대표적으로 다음과 같은 방법이 있음: SelectKBest/ SelectPercentile/ chi2
f_classif / mutual_info_classif
SelectKBest는

## **(5) Correlation Matrix를 이용한 Feature Selection**
- feature 사이의 Pearson correlation coefficient를 분석하여 서로 비슷한 정보를 가지고 있는 중복 feature를 제거
- 1에 가까움 → 강한 양의 상관관계
- -1에 가까움 → 강한 음의 상관관계
- 0에 가까움 → 선형적인 상관관계가 약함
- feature 간 상관관계가 지나치게 높다면 동일하거나 유사한 정보를 중복해서 사용하고 있을 가능성이 있으므로 하나의 feature를 제거할 수 있음
- 따라서 dropped third column from the original dataset

## **(6) Wrapper Method - Forward Selection**
- Wrapper Method는 실제 머신러닝 모델을 학습시키면서 모델 성능을 기준으로 feature subset을 선택하는 방법
- Forward Selection은 처음에는 feature가 하나도 없는 상태에서 시작하여 모델 성능을 가장 많이 향상시키는 feature를 하나씩 추가
- X_train.columns[list(sfs1.k_feature_idx_)] -> Index(['OverallQual', 'YearRemodAdd', 'TotalBsmtSF', '2ndFlrSF', 'GrLivArea',
       'BsmtFullBath', 'BsmtHalfBath', 'Fireplaces', 'GarageCars',
       'EnclosedPorch'],
      dtype='object')
- We can see that forward feature selection results in the above columns being selected from all the given columns

## **(7) Wrapper Method - Backward Elimination**
- Backward Elimination은 Forward Selection과 반대로 처음에 모든 feature를 사용한 상태에서 시작
- 이후 모델 성능에 가장 적은 영향을 주는 feature를 하나씩 제거하면서 최종적인 feature subset을 구성
- 이를 통해 feature selection 방법에 따라 선택되는 변수와 모델 성능이 달라질 수 있음을 알 수 있음
- X_train.columns[list(sfs1.k_feature_idx_)] -> Index(['MSSubClass', 'OverallQual', 'OverallCond', 'MasVnrArea', 'BsmtFinSF1',
       'TotalBsmtSF', 'GrLivArea', 'BsmtFullBath', 'GarageCars', 'PoolArea'],
      dtype='object')
- So, backward feature elimination results in the following columns being selected

## **(8) Embedded Method - LASSO**
- Embedded Method는 feature selection이 모델 학습 과정에 포함되는 방법
- 대표적인 방법으로 LASSO와 Ridge가 있으며, 특히 LASSO는 L1 Regularization을 사용하여 일부 feature의 coefficient를 정확히 0으로 만들 수 있음
- Coefficient가 0이 된 feature는 모델에서 제거할 수 있기 때문에 feature selection에 활용할 수 있음
- 37개의 feature 중 33개를 선택하고 4개의 coefficient를 0으로 축소. 따라서 alpha가 너무 크면 중요한 feature까지 제거할 수 있기 때문에 적절한 regularization 강도를 선택하는 것이 중요

## **(9) Embedded Method - Random Forest Feature Importance**
- Random Forest는 여러 개의 Decision Tree를 결합한 모델이며, 각 feature가 의사결정 과정에서 얼마나 중요한지를 feature importance로 확인할 수 있음
- 이를 통해 모델의 예측에 중요한 변수를 확인할 수 있을 뿐만 아니라, 중요도가 낮은 feature를 제거하여 모델을 단순화할 수도 있음

## **(10) 최종 모델 결정**
- Correlation Statistics
- Selection Method
- Transform Variables
- 4 best ways of Feature Selection


# 3-1. 주요 코드




In [ ]:
# (1) 상수형 변수 제거
from sklearn.feature_selection import VarianceThreshold

sel = VarianceThreshold(threshold=0)
sel.fit(X_train)

X_train = sel.transform(X_train)
X_test = sel.transform(X_test)

sel = VarianceThreshold(threshold=0.01)
sel.fit(X_train)

X_train = sel.transform(X_train)
X_test = sel.transform(X_test)

In [ ]:
# (2) Random Forest의 feature importance
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

sfs1 = SFS(RandomForestRegressor(),
           k_features=10,
           forward=True,
           floating=False,
           verbose=2,
           scoring='r2',
           cv=3)

sfs1 = sfs1.fit(np.array(X_train), y_train)

In [ ]:
# (3) Lasso를 이용한 Embedded Feature Selection
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numerical_vars = list(data.select_dtypes(include=numerics).columns)
data = data[numerical_vars]

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(labels=['SalePrice'], axis=1),
    data['SalePrice'],
    test_size=0.3,
    random_state=0)

sel_ = SelectFromModel(Lasso(alpha=100))
sel_.fit(scaler.transform(X_train.fillna(0)), y_train)

sel_.get_support()

# 4. 새롭게 알게 된 내용 / 어려운 내용 / 배울 점
- 변수의 개수가 많아지면 모델의 복잡도가 증가하고 불필요하거나 중복되는 정보가 포함될 수 있기 때문에, 적절한 feature selection을 통해 모델을 단순화하고 과적합을 줄이는 것의 중요성을 느꼈다. 특히 Filter, Wrapper, Embedded 방식의 차이를 이해할 수 있었다. Filter 방법은 모델을 학습시키기 전에 통계적인 기준으로 변수를 빠르게 제거하기 때문에 계산 비용이 낮다. Wrapper 방법은 실제 모델의 성능을 기준으로 feature subset을 선택하기 때문에 더 직접적인 방법이지만 계산량이 많다. Embedded 방법은 모델 학습 과정에서 feature selection이 함께 이루어진다는 차이가 있다.

- 또한 상수 변수와 준상수 변수의 차이도 새롭게 알게 되었다. 상수 변수는 모든 관측치에서 같은 값을 가지므로 예측에 아무런 정보를 주지 않는다. 반면 준상수 변수는 대부분의 값이 동일하지만 일부 다른 값이 존재한다. 따라서 준상수 변수는 무조건 제거하기보다는 실제 모델 성능에 미치는 영향을 확인할 필요가 있다.

- 어려웠던 부분은 어떤 feature selection 방법을 선택해야 하는지 판단하는 것이었다. 데이터의 종류와 타깃 변수의 형태에 따라 chi2, ANOVA, Mutual Information, 상관계수 등의 방법이 달라지고, 같은 데이터에서도 선택 방법에 따라 결과가 달라질 수 있기 때문이다. 따라서 특정 방법이 항상 최고의 방법이라고 생각하기보다는 여러 방법을 적용하고 교차검증 등을 통해 성능을 비교해야 한다는 것을 배웠다.